# OneIE + mBERT — GENEVA



In [ ]:
import torch, socket
print(torch.__version__, 'CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU session required'
print(torch.cuda.get_device_name(0))
with socket.create_connection(('pypi.org',443), timeout=15):
    print('Network OK')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2','tokenizers==0.19.1','numpy<2','tqdm','nltk','lxml'],check=True)


In [ ]:
from pathlib import Path
import json
candidates = list(Path('/kaggle/input').rglob('run_oneie_dataset.py'))
assert len(candidates) == 1, candidates
INPUT = candidates[0].parent.parent
# Apply zero-score summary fix without modifying the read-only input Dataset.
import shutil
FIXED = Path('/kaggle/working/geneva-fixed-input')
shutil.copytree(INPUT, FIXED, dirs_exist_ok=True)
util = FIXED / 'OneIE/util.py'
util.write_text(util.read_text().replace('best_dev_score = 0', 'best_dev_score = -1'))
INPUT = FIXED
candidates = [INPUT / 'scripts/run_oneie_dataset.py']
print((INPUT / 'native/conversion_report.json').read_text())
subprocess.run([sys.executable,'-u',str(candidates[0]),'--input',str(INPUT),'--dataset','GENEVA'],check=True)


In [ ]:
WORK = Path('/kaggle/working/oneie-geneva')
log = sorted((WORK/'runs').glob('*/log.txt'))[-1]
rows=[json.loads(line) for line in log.read_text().splitlines()[1:]]
best=max(rows,key=lambda row:row['dev']['role']['f'])
print('Best epoch by projected dev Role-C:', best['epoch'])
print('Projected gold scores:', json.dumps(best,indent=2))
print('Unpruned token-aligned gold scores:', (log.parent/'full_gold_scores.json').read_text())
print('Checkpoints and predictions:', log.parent)
